# word2manylanguages: one-language pipeline, step by step

Each `# %%` block below is one step, runnable on its own (VS Code / Jupyter
"Run Cell") so you can inspect output between steps rather than running the
whole thing blind. Every step reuses the underlying function's own
skip-if-exists behavior (pass overwrite=True on any call to force a redo).

Set `language` and `version` in the Setup cell, then run cells top to
bottom. tw (Traditional Chinese) is a special case handled inline below --
see its cells' comments.

version: '2018' (default -- matches every already-published DOI; only
meaningful for the subtitles side, since Wikipedia has no dated-vintage
concept and is shared/reused either way) or '2024' (the newer OpenSubtitles
add-on corpus -- the only option for languages with no 2018 data at all).

In [1]:
import os
import sys

HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else os.getcwd()
sys.path.insert(0, os.path.join(HERE, "01_corpus_preprocessing"))
sys.path.insert(0, os.path.join(HERE, "02_model_training"))
sys.path.insert(0, os.path.join(HERE, "eval_inputs"))

import corpus_preprocessing as cp
import model_training as mt
import build_counts_tokenized as bc

cp.basedir = mt.basedir = bc.basedir = HERE

language = "af"     # two-letter code, e.g. 'af'
version = "2018"    # '2018' or '2024' -- see module docstring above

# the (possibly version-suffixed) key used for every subtitles-side and
# downstream (corpus/counts/model) filename -- e.g. 'en' for 2018, 'en-2024'
# for 2024. Wikipedia-side filenames always use the bare `language`, never this.
subs_key = language if version == "2018" else f"{language}-{version}"

## 1. Download raw data
in:  (network) Wikimedia dump, OpenSubtitles
out: raw/wikipedia-{language}.bz2
     raw/subtitles-{subs_key}.zip

In [ ]:
if language == "tw":
    print("tw has no real Wikipedia of its own (see cell 3's comment below) -- skipping wikipedia download.")
elif version == "2018":
    print("no download for 2018 Wikipedia, since we already have it from student's download.")
else:
    cp.download("wikipedia", language, version=version)

In [2]:
cp.download("subtitles", language, version=version)

Remote file https://object.pouta.csc.fi/OPUS-OpenSubtitles/2018/raw/af.zip, Local file subtitles-af.zip
Download complete.


## 2. Clean + prune wikipedia (skip this pair entirely for tw -- see cell 3)
in:  raw/wikipedia-{language}.bz2
out: preprocessed/wikipedia-{language}-pre.zip (clean)
     preprocessed/wikipedia-{language}-pruned.zip (prune -- document-level
     dedup only, e.g. the same movie/article re-uploaded under a different
     ID; never touches sentence/phrase content within or across distinct
     documents)

In [3]:
if language != "tw":
    cp.clean_wikipedia(language)

Preprocessing af Wikipedia dump.
Complete


In [4]:
if language != "tw":
    cp.prune("wikipedia", language)

Checking for duplicates.


Big bucket found. key:18427e:0, len:33116
Big bucket found. key:cc767:1, len:33117
Big bucket found. key:3a6002:2, len:33116
Big bucket found. key:aec11:0, len:251
Big bucket found. key:aec11:0, len:251
Big bucket found. key:aec11:0, len:251
Big bucket found. key:9faa2:1, len:235
Big bucket found. key:9faa2:1, len:235
Big bucket found. key:aec11:0, len:251
Big bucket found. key:9faa2:1, len:235
Big bucket found. key:aec11:0, len:251
Big bucket found. key:aec11:0, len:251
Big bucket found. key:9faa2:1, len:235
Big bucket found. key:aec11:0, len:251
Big bucket found. key:aec11:0, len:251
Big bucket found. key:9faa2:1, len:235
Big bucket found. key:aec11:0, len:251
Big bucket found. key:aec11:0, len:251
Big bucket found. key:aec11:0, len:251
Big bucket found. key:aec11:0, len:251
Big bucket found. key:9faa2:1, len:235
Big bucket found. key:9faa2:1, len:235
Big bucket found. key:9faa2:1, len:235
Big bucket found. key:9faa2:1, len:235
Big bucket found. key:aec11:0, len:251
Big bucket found.

(wiki-af-10.txt, wiki-af-33572.txt)
(wiki-af-10.txt, wiki-af-27839.txt)
(wiki-af-10.txt, wiki-af-138578.txt)
(wiki-af-10.txt, wiki-af-73433.txt)
(wiki-af-10.txt, wiki-af-112856.txt)
(wiki-af-10.txt, wiki-af-67718.txt)
(wiki-af-10.txt, wiki-af-158168.txt)
(wiki-af-10.txt, wiki-af-80495.txt)
(wiki-af-10.txt, wiki-af-132969.txt)
(wiki-af-10.txt, wiki-af-19140.txt)
(wiki-af-10.txt, wiki-af-3181.txt)
(wiki-af-10.txt, wiki-af-58686.txt)
(wiki-af-10.txt, wiki-af-89459.txt)
(wiki-af-10.txt, wiki-af-24445.txt)
(wiki-af-10.txt, wiki-af-142275.txt)
(wiki-af-10.txt, wiki-af-115401.txt)
(wiki-af-10.txt, wiki-af-16922.txt)
(wiki-af-10.txt, wiki-af-130937.txt)
(wiki-af-10.txt, wiki-af-38791.txt)
(wiki-af-10.txt, wiki-af-105353.txt)
(wiki-af-10.txt, wiki-af-10050.txt)
(wiki-af-10.txt, wiki-af-54401.txt)
(wiki-af-10.txt, wiki-af-8959.txt)
(wiki-af-10.txt, wiki-af-7582.txt)
(wiki-af-10.txt, wiki-af-54940.txt)
(wiki-af-10.txt, wiki-af-33480.txt)
(wiki-af-10.txt, wiki-af-49034.txt)
(wiki-af-10.txt, wiki-a

Big bucket found. key:9faa2:1, len:235
Big bucket found. key:aec11:0, len:251
Big bucket found. key:aec11:0, len:251
Big bucket found. key:aec11:0, len:251
Big bucket found. key:9faa2:1, len:235
Big bucket found. key:aec11:0, len:251
Big bucket found. key:9faa2:1, len:235
Big bucket found. key:9faa2:1, len:235
Big bucket found. key:aec11:0, len:251
Big bucket found. key:9faa2:1, len:235
Big bucket found. key:9faa2:1, len:235
Big bucket found. key:9faa2:1, len:235
Big bucket found. key:9faa2:1, len:235
Big bucket found. key:9faa2:1, len:235
Big bucket found. key:9faa2:1, len:235
Big bucket found. key:9faa2:1, len:235
Big bucket found. key:9faa2:1, len:235
Big bucket found. key:9faa2:1, len:235
Big bucket found. key:aec11:0, len:251
Big bucket found. key:aec11:0, len:251
Big bucket found. key:9faa2:1, len:235
Big bucket found. key:aec11:0, len:251
Big bucket found. key:aec11:0, len:251
Big bucket found. key:9faa2:1, len:235
Big bucket found. key:9faa2:1, len:235
Big bucket found. key:aec

(wiki-af-1389.txt, wiki-af-2117.txt)
(wiki-af-1389.txt, wiki-af-2112.txt)
(wiki-af-1389.txt, wiki-af-2688.txt)
(wiki-af-1389.txt, wiki-af-2351.txt)
(wiki-af-1389.txt, wiki-af-2808.txt)
(wiki-af-1391.txt, wiki-af-2298.txt)
(wiki-af-1391.txt, wiki-af-1335.txt)
(wiki-af-1391.txt, wiki-af-2662.txt)
(wiki-af-1391.txt, wiki-af-1387.txt)
(wiki-af-1391.txt, wiki-af-2326.txt)
(wiki-af-1394.txt, wiki-af-2763.txt)
(wiki-af-1394.txt, wiki-af-1696.txt)
(wiki-af-1395.txt, wiki-af-2100.txt)
(wiki-af-1395.txt, wiki-af-2565.txt)
(wiki-af-1395.txt, wiki-af-1556.txt)
(wiki-af-1398.txt, wiki-af-479.txt)
(wiki-af-1399.txt, wiki-af-2025.txt)
(wiki-af-1399.txt, wiki-af-1411.txt)
(wiki-af-1399.txt, wiki-af-2169.txt)
(wiki-af-1399.txt, wiki-af-930.txt)
(wiki-af-1399.txt, wiki-af-2646.txt)
(wiki-af-1401.txt, wiki-af-2027.txt)
(wiki-af-1524.txt, wiki-af-2139.txt)
(wiki-af-1525.txt, wiki-af-1594.txt)
(wiki-af-1526.txt, wiki-af-1788.txt)
(wiki-af-1526.txt, wiki-af-1640.txt)
(wiki-af-1526.txt, wiki-af-2067.txt)
(wi

...

(wiki-af-82885.txt, wiki-af-82930.txt)
(wiki-af-82885.txt, wiki-af-82909.txt)
(wiki-af-82885.txt, wiki-af-82890.txt)
(wiki-af-82885.txt, wiki-af-82952.txt)
(wiki-af-82886.txt, wiki-af-82931.txt)
(wiki-af-82887.txt, wiki-af-82929.txt)
(wiki-af-82887.txt, wiki-af-82909.txt)
(wiki-af-82888.txt, wiki-af-82931.txt)
(wiki-af-82888.txt, wiki-af-82957.txt)
(wiki-af-82889.txt, wiki-af-82941.txt)
(wiki-af-82889.txt, wiki-af-82895.txt)
(wiki-af-82889.txt, wiki-af-76501.txt)
(wiki-af-82889.txt, wiki-af-76503.txt)
(wiki-af-82896.txt, wiki-af-82895.txt)
(wiki-af-82896.txt, wiki-af-82925.txt)
(wiki-af-82910.txt, wiki-af-82968.txt)
(wiki-af-82910.txt, wiki-af-82973.txt)
(wiki-af-82910.txt, wiki-af-82930.txt)
(wiki-af-82910.txt, wiki-af-82971.txt)
(wiki-af-82910.txt, wiki-af-82890.txt)
(wiki-af-82917.txt, wiki-af-82931.txt)
(wiki-af-82917.txt, wiki-af-82956.txt)
(wiki-af-82917.txt, wiki-af-82890.txt)
(wiki-af-82918.txt, wiki-af-82951.txt)
(wiki-af-82918.txt, wiki-af-82673.txt)
(wiki-af-82922.txt, wiki-

Big bucket found. key:cc767:1, len:33117


(wiki-af-98824.txt, wiki-af-98825.txt)
(wiki-af-98893.txt, wiki-af-98903.txt)
(wiki-af-98893.txt, wiki-af-98896.txt)
(wiki-af-98893.txt, wiki-af-98895.txt)
(wiki-af-98893.txt, wiki-af-98904.txt)
(wiki-af-98894.txt, wiki-af-98896.txt)
(wiki-af-98894.txt, wiki-af-98900.txt)
(wiki-af-98894.txt, wiki-af-98904.txt)
(wiki-af-98897.txt, wiki-af-98903.txt)
(wiki-af-98897.txt, wiki-af-98898.txt)
(wiki-af-98901.txt, wiki-af-98896.txt)
(wiki-af-98901.txt, wiki-af-98900.txt)
(wiki-af-98901.txt, wiki-af-98904.txt)
(wiki-af-98906.txt, wiki-af-98919.txt)
(wiki-af-98909.txt, wiki-af-98922.txt)
(wiki-af-99962.txt, wiki-af-107586.txt)
(wiki-af-99964.txt, wiki-af-108374.txt)
(wiki-af-99964.txt, wiki-af-108384.txt)
(wiki-af-99964.txt, wiki-af-108383.txt)
(wiki-af-107481.txt, wiki-af-25402.txt)
(wiki-af-107481.txt, wiki-af-25192.txt)
(wiki-af-107557.txt, wiki-af-107571.txt)
(wiki-af-107557.txt, wiki-af-107572.txt)
(wiki-af-107557.txt, wiki-af-107576.txt)
(wiki-af-107557.txt, wiki-af-107559.txt)
(wiki-af-10

Big bucket found. key:368702:2, len:239
Big bucket found. key:d6d24:1, len:490


(wiki-af-111030.txt, wiki-af-111031.txt)
(wiki-af-114326.txt, wiki-af-114327.txt)
(wiki-af-114440.txt, wiki-af-114441.txt)
(wiki-af-115032.txt, wiki-af-115036.txt)
(wiki-af-116205.txt, wiki-af-116213.txt)
(wiki-af-116205.txt, wiki-af-116217.txt)
(wiki-af-118496.txt, wiki-af-118739.txt)
(wiki-af-118496.txt, wiki-af-118499.txt)
(wiki-af-118496.txt, wiki-af-118738.txt)
(wiki-af-118496.txt, wiki-af-118507.txt)
(wiki-af-118512.txt, wiki-af-118738.txt)
(wiki-af-118512.txt, wiki-af-118626.txt)
(wiki-af-118512.txt, wiki-af-118739.txt)
(wiki-af-118524.txt, wiki-af-118507.txt)
(wiki-af-118524.txt, wiki-af-118614.txt)
(wiki-af-118525.txt, wiki-af-118738.txt)
(wiki-af-118525.txt, wiki-af-118626.txt)
(wiki-af-118525.txt, wiki-af-118562.txt)
(wiki-af-118624.txt, wiki-af-118738.txt)
(wiki-af-118747.txt, wiki-af-118738.txt)
(wiki-af-118747.txt, wiki-af-118499.txt)
(wiki-af-118806.txt, wiki-af-118823.txt)
(wiki-af-120270.txt, wiki-af-120271.txt)
(wiki-af-120270.txt, wiki-af-120272.txt)
(wiki-af-120274.

## 3. tw only: materialize wikipedia data from zh instead of downloading
tw (Traditional Chinese / Taiwan) has no Wikipedia of its own -- ISO 639-1
"tw" is Twi, an unrelated Ghanaian language. Chinese Wikipedia only exists
as the single "zh" wiki (mixed simplified/traditional per article as each
editor wrote it). This converts zh's already-cleaned wiki text to
Taiwan-standard Traditional Chinese via OpenCC (script AND phrasing, e.g.
"software" -> 軟體 not 软件/軟件), so every step after this treats tw as if
it had legitimate wiki data all along.
in:  preprocessed/wikipedia-zh-pruned.zip (zh's cells 2 must already be done)
out: preprocessed/wikipedia-tw-pruned.zip

In [5]:
if language == "tw":
    bc.materialize_tw_wikipedia_pruned()

## 4. Clean + prune subtitles
in:  raw/subtitles-{subs_key}.zip
out: preprocessed/subtitles-{subs_key}-pre.zip (clean)
     preprocessed/subtitles-{subs_key}-pruned.zip (prune, document-level only)

In [7]:
cp.clean_subtitles(language, version=version)

BadZipFile: File is not a zip file

In [ ]:
cp.prune("subtitles", subs_key)

## 5. Concatenate into the training corpus
in:  preprocessed/wikipedia-{language}-pruned.zip
     preprocessed/subtitles-{subs_key}-pruned.zip
out: corpora/corpus-{subs_key}.txt  (one sentence per line, what
     02_model_training actually trains on)

In [ ]:
cp.concatenate_corpus(language, version=version)

## 6. Build frequency counts (this project's own corpus, not an external mirror)
in:  preprocessed/wikipedia-{language}-pruned.zip
     preprocessed/subtitles-{subs_key}-pruned.zip
out: eval_inputs/counts/dedup.{language}wiki-meta.words.unigrams.tsv.zip
     eval_inputs/counts/dedup.{subs_key}.words.unigrams.tsv.zip

In [ ]:
if language == "tw":
    bc.build_tw_wiki_counts()  # derives from zh, same as cell 3
else:
    bc.count_unigrams("wikipedia", language)

In [ ]:
bc.count_unigrams("subtitles", subs_key)

## 7. Train models -- 60 configs (dim: 50/100/200/300/500, window: 1-6, algo: cbow/sg)
in:  corpora/corpus-{subs_key}.txt
out: models/{subs_key}_{dim}_{window}_{algo}_wxd.csv.bz2 x 60
This is the slow step. To try just one configuration first, shrink the
sweep before calling build_models (see 02_model_training/README.md):
  mt.dimension_list = [50]; mt.window_list = [1]; mt.algo_list = ['cbow']

In [ ]:
mt.build_models(subs_key)

## 8. Upload to Zenodo (separate script, not run automatically -- inspect
the models first). See download/zenodo_upload.py's module docstring.

  python download/zenodo_upload.py --language {language} --version {version} \
      --models-dir models/ --dry-run